# Lab Day 1 — Notebook (PSEUDO-CODE)

> **Đây là khung pseudo-code.** Mọi ô có `# TODO` là phần bạn phải tự viết. Hãy giữ cấu trúc mục (Part 0–4) và
> viết dự đoán/nhận xét bằng chữ của bạn. Đọc `README.md`, `GUIDE.md`, `RUBRIC.md` trước.
>
> **Quy trình làm bài**
> 1. `git pull` repo, tạo thư mục nộp: copy cả thư mục `code/` vào `submission_<MSSV>/code/`.
> 2. Mở `submission_<MSSV>/code/lab.ipynb` và làm việc **tại đó** (nên `REPO_ROOT = "../.."` đúng và ảnh/kết quả ghi vào `submission_<MSSV>/figures`, `results`).
> 3. Ở Colab: chạy ô clone repo bên dưới, rồi làm tương tự (đặt `REPO_ROOT` cho đúng).
> 4. Cuối cùng: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [ ]:
# Colab bootstrap: mount Drive and run from the repo's code/ folder so REPO_ROOT=".." and OUT_DIR=".." resolve
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB: from google.colab import drive; drive.mount("/content/drive")
if IN_COLAB: os.chdir("/content/drive/MyDrive/K4-Track4-Day1-Neuralnetwork/code")
if os.getcwd() not in sys.path: sys.path.insert(0, os.getcwd())
%load_ext autoreload
%autoreload 2
print(os.getcwd())

In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess, math
import numpy as np, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

REPO_ROOT = ".."        # thư mục gốc repo (chứa data/ và scripts/), tính từ code/
OUT_DIR   = ".."        # nơi ghi figures/, results/, experiments.xlsx, predictions_eval.csv (= gốc repo)

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"torch {torch.__version__} | device = {device}" + (f" | GPU = {torch.cuda.get_device_name(0)}" if device == "cuda" else ""))
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx

## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [ ]:
# Part 0: tạo train.npz / eval.npz từ metadata (script tự kiểm tra tính toàn vẹn; chạy lại cho kết quả y hệt)
r = subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True, capture_output=True,
                   text=True, encoding="utf-8", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print(r.stdout)

# Tách val 20% từ train (phân tầng, seed 42), chuẩn hoá 10 cột số bằng mean/std của X_tr, đưa lên device
data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")
assert (len(data["X_tr"]), len(data["X_val"]), len(data["X_eval"])) == (371_847, 92_962, 116_203)

num = data["X_tr"][:, :10]
print("\nmean 10 cột số trên X_tr:", np.round(num.mean(0).cpu().numpy(), 4))
print("std  10 cột số trên X_tr:", np.round(num.std(0).cpu().numpy(), 4))
assert num.mean(0).abs().max() < 1e-3 and (num.std(0) - 1).abs().max() < 1e-3
assert set(torch.unique(data["X_tr"][:, 10:]).tolist()) <= {0.0, 1.0}   # 44 cột nhị phân giữ nguyên

**Nhận xét Part 0:** val được tách **từ train** (20%, phân tầng, seed 42), nên tỉ lệ lớp ở train/val/eval gần như trùng nhau.
Mean/std chỉ tính trên `X_tr`. Nếu tính cả trên val/eval, thống kê chuẩn hoá đã "nhìn thấy" chính dữ liệu dùng để đo (rò rỉ thông tin), nên điểm val/eval không còn đo đúng khả năng trên dữ liệu mới.
`X_eval` chỉ được biến đổi bằng mean/std đó và không tham gia chọn cấu hình hay dừng sớm. Lô cuối của mỗi epoch được giữ lại dù nhỏ hơn (371 847 = 726·512 + 135).
Mốc thấp nhất phải vượt: luôn đoán lớp đa số → accuracy ≈ 0,4876 trên val.

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [ ]:
# Part 1a/1b: M-base, số tham số và shape sau từng lớp
set_seed(1)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print("số tham số:", n_params)
assert n_params == count_params(model) == EXPECTED_PARAMS[(256, 128)] == 47_879

with torch.no_grad():
    h = torch.randn(8, 54, device=device)
    print(f"{'input':>8} -> {tuple(h.shape)}")
    for layer in model.net:
        h = layer(h)
        print(f"{type(layer).__name__:>8} -> {tuple(h.shape)}")
assert h.shape == (8, 7)

In [ ]:
# Part 1c-1: loss bước 0 trên val (eval mode, chưa cập nhật bước nào) phải ≈ ln 7
model.eval()
with torch.no_grad():
    logits0 = model(data["X_val"])
    step0_loss = F.cross_entropy(logits0, data["y_val"]).item()
print(f"loss bước 0 trên val = {step0_loss:.4f} | ln 7 = {math.log(7):.4f} | chênh = {step0_loss - math.log(7):+.4f}")
print(f"std của logit bước 0 = {logits0.std().item():.3f}")
print("std kích hoạt sau mỗi Linear (z1, z2, logits), lô val 4096 mẫu:",
      [round(s, 3) for s in activation_stats(model, data["X_val"][:4096])])

In [ ]:
# Part 1c-2: quá khớp 20 mẫu (dropout tắt). Loss phải về gần 0 và accuracy 100% trên 20 mẫu đó
set_seed(1)
idx = torch.randperm(len(data["X_tr"]), generator=torch.Generator().manual_seed(0))[:20].to(device)
x20, y20 = data["X_tr"][idx], data["y_tr"][idx]
tiny = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
opt = torch.optim.Adam(tiny.parameters(), lr=1e-2)
losses = []
for step in range(300):
    loss = F.cross_entropy(tiny(x20), y20)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    losses.append(loss.item())
with torch.no_grad():
    acc20 = (tiny(x20).argmax(1) == y20).float().mean().item()
print(f"nhãn của 20 mẫu: {torch.bincount(y20, minlength=7).tolist()}")
print(f"loss bước 0 = {losses[0]:.4f}, bước {len(losses) - 1} = {losses[-1]:.2e}; accuracy trên 20 mẫu = {acc20:.0%}")

plt.figure(figsize=(6, 3.5))
plt.semilogy(losses)
plt.xlabel("bước"); plt.ylabel("cross-entropy (log)")
plt.title("Quá khớp 20 mẫu: M-base, Adam lr=1e-2, dropout 0")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
assert losses[-1] < 1e-2 and acc20 == 1.0

In [ ]:
# Part 1d: sau một lần backward, mọi tham số phải có gradient khác None và khác 0
model.train()
model.zero_grad(set_to_none=True)
F.cross_entropy(model(data["X_tr"][:512]), data["y_tr"][:512]).backward()
for name, (pname, p) in zip(["W1", "b1", "W2", "b2", "W3", "b3"], model.named_parameters()):
    assert p.grad is not None, f"{name}: grad là None"
    gn = p.grad.norm().item()
    print(f"{name}  {pname:<12} {str(tuple(p.shape)):<11} grad norm = {gn:.4e}")
    assert gn > 0, f"{name}: grad bằng 0"

**Nhận xét Part 1 (tự viết):** loss bước 0 đo được = ___, so với ln 7 = 1.946 vì sao ...; quá khớp 20 mẫu: ...

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [ ]:
# TODO: chọn lr cho baseline bằng VAL (chạy vài giá trị, ví dụ quanh 0.01 -> 0.1); ghi bảng lr -> val_macro_f1
# TODO: cfg = {**DEFAULT_CFG, "lr": <lr đã chọn>}
# TODO: chạy baseline với >= 2 seed (khuyến khích 3): result = run_experiment({**cfg, "seed": s, "exp_id": f"base-s{s}"}, data)
# TODO: với mỗi result: save_result(result, f"{OUT_DIR}/results"); plot_run(result, f"{OUT_DIR}/figures/{exp_id}.png")
# TODO: tính trung bình ± độ lệch chuẩn (val_macro_f1, val_acc) qua các seed -> ngưỡng nhiễu 2σ

**Nhận xét baseline (tự viết):** hình dạng đường cong (còn giảm? bắt đầu quá khớp?), val_acc so với mốc "đoán đa số", độ nhiễu giữa các seed ...

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Thí nghiệm `<exp_id>` — chủ đề `<nhóm>`
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [ ]:
# Mẫu một thí nghiệm. Lặp lại / bọc trong vòng for qua danh sách cfg bạn tự thiết kế.
# TODO: exp_cfg = {**cfg, "exp_id": "...", "group": "...", "description": "...", <chỉ đổi MỘT yếu tố>}
# TODO: result = run_experiment(exp_cfg, data)
# TODO: save_result(...); plot_run(...)   # mỗi thí nghiệm một ảnh figures/<exp_id>.png
# TODO: in summary (best_val_loss, val_acc, val_macro_f1, ...) và so với baseline + ngưỡng nhiễu 2σ

**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

In [ ]:
# TODO: ảnh chồng theo nhóm: plot_compare(list_of_results_in_group, "val_loss", f"{OUT_DIR}/figures/compare_<nhóm>.png")

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [ ]:
# TODO: chọn cấu hình cuối cùng theo VAL (có thể là baseline nếu bạn không kết hợp thêm gì)
# TODO: final_eval(cfg_final, result_final, data, f"{OUT_DIR}/predictions_eval.csv")
# TODO: chạy: python scripts/evaluate.py --pred <OUT_DIR>/predictions_eval.csv --out <OUT_DIR>/eval_result.json (từ REPO_ROOT)
# TODO: đọc eval_result.json: ghi accuracy, macro_f1 vào bảng và báo cáo; làm tương tự cho baseline (predictions riêng, không nộp nếu không cần)

In [ ]:
# TODO: phân tích lỗi trên eval: bảng precision/recall/F1 theo lớp (từ eval_result.json) và ma trận nhầm lẫn;
#       lớp nào khó nhất? vì sao (số mẫu ít? nhầm với lớp nào?)  -> viết nhận xét ở ô markdown bên dưới

**Phân tích lỗi (tự viết):** ...

In [ ]:
# TODO: results = load_results(f"{OUT_DIR}/results"); rows = [to_row(r, ...) for r in results]
# TODO: write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
# TODO: bổ sung sheet Seeds (exp_id các lần chạy baseline) và nhận xét ở sheet Summary